## OpenAI News Scraper

In [13]:
# [Updated] OpenAI Scraper using curl_cffi for Cloudflare Bypass
try:
    from curl_cffi import requests
except ImportError:
    print("Error: curl_cffi not installed. Please run: pip install curl-cffi")
    import requests

from bs4 import BeautifulSoup
from datetime import datetime, timedelta
import time
import re

# ==========================================
# OpenAI Scraper Helpers
# ==========================================

def parse_openai_date(date_str):
    """Parse dates like 'Dec 19, 2025', 'October 31, 2024', etc."""
    if not date_str: return None
    date_str = date_str.strip()
    formats = [
        '%b %d, %Y',       # Dec 19, 2025
        '%B %d, %Y',       # October 31, 2024
        '%Y-%m-%d',
    ]
    for fmt in formats:
        try:
            return datetime.strptime(date_str, fmt)
        except ValueError:
            continue
    return None

def normalize_date(d):
    if isinstance(d, str):
        for fmt in ['%Y-%m-%d', '%b %d, %Y', '%B %d, %Y']:
             try:
                 return datetime.strptime(d, fmt)
             except:
                 continue
    return d

def get_openai_news_list(start_date, end_date=None):
    """
    Scrapes article list from https://openai.com/news/
    """
    url = "https://openai.com/news/"
    
    try:
        start_dt = normalize_date(start_date)
        end_dt = normalize_date(end_date) if end_date else datetime.now()
        if start_dt: start_dt = start_dt.replace(hour=0, minute=0, second=0)
        if end_dt: end_dt = end_dt.replace(hour=23, minute=59, second=59)
        
        print(f"Fetching OpenAI News List... Range: {start_dt} ~ {end_dt}")
        
        # Use curl_cffi to impersonate Chrome
        response = requests.get(url, impersonate="chrome", timeout=30)
        
        if response.status_code != 200:
            print(f"Failed to access list page: {response.status_code}")
            return []
            
        soup = BeautifulSoup(response.content, 'html.parser')
        articles = []
        
        all_links = soup.find_all('a', href=True)
        
        for link in all_links:
            href = link['href']
            if not (href.startswith('/news/') or href.startswith('/index/')):
                continue
            if len(href) < 8: continue 
            
            full_url = f"https://openai.com{href}" if href.startswith('/') else href
            
            card_text = link.get_text("\n", strip=True)
            texts = card_text.split("\n")
            
            found_date = None
            found_date_str = ""
            title_candidate = ""
            
            for t in texts:
                d = parse_openai_date(t)
                if d:
                    found_date = d
                    found_date_str = t
                    break
            
            h3 = link.find('h3')
            if h3:
                title_candidate = h3.get_text(strip=True)
            else:
                candidates = [line for line in texts if line != found_date_str and len(line) > 10]
                if candidates: 
                    title_candidate = candidates[0]
                else:
                    title_candidate = "No Title Found"
            
            if found_date:
                if start_dt <= found_date <= end_dt:
                    if not any(a['url'] == full_url for a in articles):
                        articles.append({
                            'url': full_url,
                            'date': found_date_str,
                            'dt': found_date,
                            'title': title_candidate
                        })
            
        print(f"Found {len(articles)} matching articles in list.")
        return articles
        
    except Exception as e:
        print(f"Error in get_openai_news_list: {e}")
        return []

def scrape_openai_article_detail(url, list_info):
    """
    Scrapes detail of a single OpenAI article.
    """
    try:
        # Use curl_cffi
        response = requests.get(url, impersonate="chrome", timeout=30)
        
        soup = BeautifulSoup(response.content, 'html.parser')
        
        content_div = soup.find('article')
        if not content_div: 
             content_div = soup.find('div', class_=lambda c: c and 'prose' in c)
        
        if not content_div:
            content_div = soup.find('main') or soup.find('body')
            
        if content_div:
            for tag in content_div(['script', 'style', 'button', 'nav', 'footer', 'header', 'form']):
                tag.decompose()
            raw_content = content_div.get_text(separator='\n', strip=True)
        else:
            raw_content = ""
            
        return {
            "raw_content": raw_content,
            "source_url": url,
            "date": list_info['date'],
            "raw_title": list_info['title'],
            "source_name": "OpenAI"
        }

    except Exception as e:
        print(f"Error scraping article {url}: {e}")
        return None

def run_openai_scraper(start_date, end_date=None):
    target_articles = get_openai_news_list(start_date, end_date)
    results = []
    
    for article in target_articles:
        print(f"  > Scraping: {article['title']} ({article['date']})")
        details = scrape_openai_article_detail(article['url'], article)
        if details:
             results.append(details)
        time.sleep(1)
        
    print(f"Done. Scraped {len(results)} articles.")
    return results


In [14]:
# Test the scraper with specific range
if 'run_openai_scraper' in globals():
    # User requested specific range
    start_date = '2026-01-01'
    end_date = '2026-01-04'
    
    print(f"Running scraper from {start_date} to {end_date}...")
    res = run_openai_scraper(start_date, end_date)
    
    print("\n[Results]")
    for r in res:
        print(r)
else:
    print("Please run the scraper definition cell first.")


Running scraper from 2026-01-01 to 2026-01-04...
Fetching OpenAI News List... Range: 2026-01-01 00:00:00 ~ 2026-01-04 23:59:59
Found 1 matching articles in list.
  > Scraping: Apply to OpenAI Grove (Jan 2, 2026)
Done. Scraped 1 articles.

[Results]
{'raw_content': 'January 2, 2026\nCompany\nApply to OpenAI Grove\nA program for individuals early in their company building journey.\nLoading…\nToday, we’re opening the applications for the next cohort of OpenAI Grove, a program for technical talent at the very start of their company-building journey. The Grove is not a startup accelerator or traditional program: it offers pre-idea individuals deeply curious about building in AI a dense talent network, co-building with OpenAI researchers, and resources designed to accelerate your journey. As participants explore early concepts, they will receive counsel from the OpenAI team and community with peers in OpenAI Grove.\nThis program is the starting point of a long-term network. It will begin wit

In [15]:
r

{'raw_content': 'January 2, 2026\nCompany\nApply to OpenAI Grove\nA program for individuals early in their company building journey.\nLoading…\nToday, we’re opening the applications for the next cohort of OpenAI Grove, a program for technical talent at the very start of their company-building journey. The Grove is not a startup accelerator or traditional program: it offers pre-idea individuals deeply curious about building in AI a dense talent network, co-building with OpenAI researchers, and resources designed to accelerate your journey. As participants explore early concepts, they will receive counsel from the OpenAI team and community with peers in OpenAI Grove.\nThis program is the starting point of a long-term network. It will begin with five weeks of content and programming hosted in the OpenAI San Francisco HQ, including in-person workshops, weekly office hours, and mentoring from OpenAI technical leaders. In addition to technical support and community, participants will also ha

In [16]:
print(r['raw_content'])

January 2, 2026
Company
Apply to OpenAI Grove
A program for individuals early in their company building journey.
Loading…
Today, we’re opening the applications for the next cohort of OpenAI Grove, a program for technical talent at the very start of their company-building journey. The Grove is not a startup accelerator or traditional program: it offers pre-idea individuals deeply curious about building in AI a dense talent network, co-building with OpenAI researchers, and resources designed to accelerate your journey. As participants explore early concepts, they will receive counsel from the OpenAI team and community with peers in OpenAI Grove.
This program is the starting point of a long-term network. It will begin with five weeks of content and programming hosted in the OpenAI San Francisco HQ, including in-person workshops, weekly office hours, and mentoring from OpenAI technical leaders. In addition to technical support and community, participants will also have the opportunity to g